# Document Retrieval


Document retrieval involves locating and retrieving documents from a large collection based on user queries. It is essential in search engines, digital libraries, and document management systems. The process includes:

- **Query Processing**: Transforming user queries for efficient processing.
- **Indexing**: Creating an index (usually an inverted index) that lists documents containing specific terms for quick retrieval.
- **Ranking and Retrieval**: Documents that match query terms are retrieved and ranked based on relevance, using measures like term frequency and document frequency.
- **Relevance Feedback**: Users' feedback on document relevance can refine search algorithms for improved results in future queries.

    <img src="https://drive.google.com/uc?export=view&id=1LGDDWV9HF_JAHq-WfA2PBD_FUCiziezu" width="350" height="320" alt="Document Search">


In [2]:
from sklearn.datasets import fetch_20newsgroups

# Fetch the dataset
newsgroups = fetch_20newsgroups(subset='all')

# Access the data
texts = newsgroups.data  # the actual newsgroup postings
target = newsgroups.target  # the category labels
target_names = newsgroups.target_names  # the names of the categories

print("Number of texts: ", len(texts))
print("Number of categories: ", len(target_names))

Number of texts:  18846
Number of categories:  20


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Setup the vectorizer with necessary preprocessing settings
vectorizer = TfidfVectorizer(
    lowercase=True,        # Convert all characters to lowercase before tokenizing
    stop_words='english',  # Remove stopwords
    max_df=0.95,           # Terms that appear in more than 95% of the documents are ignored
    min_df=2,              # Terms that appear in less than 2 documents are ignored
    max_features=10000     # Only consider the top 10,000 features ordered by term frequency across the corpus
)

# Apply vectorization to the text data
tfidf_matrix = vectorizer.fit_transform(texts)

print(tfidf_matrix.shape)  # Output the shape of the TF-IDF matrix

(18846, 10000)


### Cosine Similarity

Cosine similarity is a metric used to measure how similar two entities (documents, vectors, etc.) are irrespective of their size. Mathematically, it calculates the cosine of the angle between two vectors projected in a multi-dimensional space. The formula is defined as:

$$
\text{Cosine Similarity} = \frac{\sum_{i=1}^{n} A_i \times B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \times \sqrt{\sum_{i=1}^{n} B_i^2}}
$$

Where:
- \(A\) and \(B\) are the vector representations of the two entities.
- \(n\) is the number of dimensions of the vectors (e.g., terms in a document).

In the context of document retrieval and recommender systems, cosine similarity can be used to assess the similarity between documents or between user preferences by comparing their vector representations (e.g., TF-IDF vectors).

The result ranges from -1 to 1:
- 1 indicates perfect similarity.
- 0 indicates no similarity.
- -1 indicates perfect dissimilarity.

This metric is particularly useful in systems where the magnitude of the vector is not relevant.

In [4]:
# @title Search based on cos similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Function to perform the query and retrieve documents
def retrieve_documents(query, vectorizer, tfidf_matrix, top_k=5):
    # Transform the query to the same vector space as the documents
    query_vec = vectorizer.transform([query])

    # Compute the cosine similarity between query vector and all document vectors
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # Get the top k documents with the highest similarity scores
    top_indices = np.argsort(-similarities)[:top_k]  # argsort returns indices of sorted array

    return top_indices, similarities[top_indices]

# Example query
query = "inflation problems"
top_indices, scores = retrieve_documents(query, vectorizer, tfidf_matrix)

# Print the results
print("Top documents for query '{}':".format(query))
for index, score in zip(top_indices, scores):
    print("Index:", index, "Score:", score, "\nDocument:", texts[index][-200:], "...\n")

Top documents for query 'inflation problems':
Index: 11693 Score: 0.33128232915162903 
Document: ver the LCIII
doesn't make a lot of sense either.  If this person is so convinced the 610
is buggy have they talked to Apple about it or are they just assuming
it's a problem with all of them?
-Terry
 ...

Index: 7856 Score: 0.32981607599674967 
Document: @cup.portal.com (Don - Hirschfeld)
Subject: Re: Toshiba 3401B CD-ROM:  Any problems?
Organization: The Portal System (TM)
Lines: 1

I have the PAS16 / Toshiba 3401 combo and have no problems with it.
 ...

Index: 7986 Score: 0.3197838158782558 
Document: t does the net think? Did the dealer just get one flaky
machine, or did Apple send the C610 out the door too early?
Is your C610 working just great, or is it buggy too?

	Jay Scott
	scott@cs.uiuc.edu
 ...

Index: 12073 Score: 0.30080798050585517 
Document: computers from Comtrade?  When I asked about 
TC, I got one reply describing problems returning a defective hard drive.
Should I expect

### BM25

BM25 is a ranking function used for document retrieval that extends the traditional TF-IDF approach. It incorporates document length normalization and a probabilistic model of term importance, based on the Inverse Document Frequency (IDF) and term frequency saturation.

Why/When Preferred:

*  Document Length Normalization - Unlike cosine similarity, BM25 adjusts for the length of a document, preventing longer documents from inherently having higher importance simply due to their size.
*  Term Frequency Saturation - It introduces a non-linear term frequency factor where, after a certain point, additional appearances of a word do not contribute as linearly to the score as initial appearances. This avoids the situation where terms that appear many times (possibly as spam) overly influence the document's relevance.

Use Cases:
*  Search Engines: BM25 is highly effective in search engines where diverse document lengths and detailed relevance scoring are critical.
*  Legal Document Retrieval: In environments where precision is more important than recall, such as retrieving the most relevant legal cases or documents based on queries that contain specific legal terminology.

In [5]:
# Install BM25 and LSI prerequisites

!pip install gensim --quiet

In [6]:
# # Fix scipy and gensim compatibility issues
# # Run this code cell once, the restart the session and comment it out before second run

# !pip install rank-bm25 --quiet
# !pip uninstall -y scipy gensim
# !pip install scipy
# !pip install gensim

In [7]:
# @title BM25
from rank_bm25 import BM25Okapi
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

# Tokenization for BM25
tokenized_corpus = [word_tokenize(doc.lower()) for doc in texts]

# Create BM25 object
bm25 = BM25Okapi(tokenized_corpus)

# Function to use BM25 to retrieve documents
def bm25_retrieve(query, top_k=5):
    query_tokens = word_tokenize(query.lower())
    scores = bm25.get_scores(query_tokens)
    top_indices = np.argsort(-scores)[:top_k]
    return top_indices, scores[top_indices]

# Example query
query = "inflation in economy"
indices, bm25_scores = bm25_retrieve(query)

# Output results
for idx, score in zip(indices, bm25_scores):
    print(f"Document index: {idx}, BM25 Score: {score}\nDocument: {texts[idx][:200]}\n")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Document index: 525, BM25 Score: 17.39233992027453
Document: From: gt0523e@prism.gatech.EDU (Michael Andre Mule)
Subject: Re: Tickets etc..
Article-I.D.: hydra.91513
Distribution: usa
Organization: Georgia Institute of Technology
Lines: 39

Let's look at the ef

Document index: 2663, BM25 Score: 16.28614829604269
Document: From: zakir@leland.Stanford.EDU (Zakir Sahul)
Subject: Inflation in car prices
Organization: DSG, Stanford University, CA 94305, USA
Distribution: usa
Lines: 5


Anyone have figures or pointers to ref

Document index: 4009, BM25 Score: 14.908646112866869
Document: From: acheng@ncsa.uiuc.edu (Albert Cheng)
Subject: Re: hard times investments was: (no subject given)
Article-I.D.: news.C52t8L.5CH
Organization: Nat'l Ctr for Supercomp App (NCSA) @ University of Ill

Document index: 453, BM25 Score: 14.739828927707077
Document: From: kjenks@gothamcity.jsc.nasa.gov
Subject: Re: Deployable Space Dock..
Organization: NASA/JSC/GM2, Space Shuttle Program Office
X-Newsreader: T

In [8]:
# Example query
query = "andromeda and stars"
indices, bm25_scores = bm25_retrieve(query)

# Output results
for idx, score in zip(indices, bm25_scores):
    print(f"Document index: {idx}, BM25 Score: {score}\nDocument: {texts[idx][-300:]}\n")

Document index: 7097, BM25 Score: 17.49165854680956
Document:  them  closer in.

If you can think of one, remember to invite me to Stockholm...

*  Steinn Sigurdsson   			Lick Observatory      	*
*  steinly@lick.ucsc.edu		"standard disclaimer"  	*
*  The laws of gravity are very,very strict			*
*  And you're just bending them for your own benefit - B.B. 1988*


Document index: 4017, BM25 Score: 15.804237288141135
Document: nd place them  closer in.

There are more than 130 GRB different models in the refereed literature.
Right now, the theorists have a sort of unofficial moratorium
on new models until new observational evidence comes in.

-- 
		David M. Palmer		palmer@alumni.caltech.edu
					palmer@tgrs.gsfc.nasa.gov


Document index: 5331, BM25 Score: 15.057514311270063
Document:  son (who was doing the broadcast with him),
"What will you do now?" responded, "First I'm going to get me a new pair of
slippers.  Then I'm going to sit in my easy chair and watch the world go by."

Thank yo

### Latent Semantic Indexing (LSI)
LSI uses a mathematical technique known as Singular Value Decomposition (SVD) on the document-term matrix to reduce its dimensionality. This reduction helps in capturing the underlying concepts or topics in the data, which might not be directly apparent from the raw term frequencies.

Why/When Preferred:

*  Handling Synonymy and Polysemy - LSI can understand different words that mean the same thing (synonymy) and the same word having multiple meanings (polysemy), unlike cosine similarity which solely relies on exact matches.
*  Conceptual Retrieval - By reducing dimensionality to capture latent concepts, LSI can find relationships between terms that are not explicitly stated, thus improving the quality of the results in concept-based searching.

Use Cases:
*  Recommender Systems: LSI can be used to recommend content by finding underlying patterns and similarities between items in a way that is not purely dependent on the explicit content.
*  Academic Research: Helps in searching through academic papers where the retrieval needs to consider the conceptual similarity between documents, not just keyword matching.

In [9]:
# @title LSI
from gensim import corpora, models, similarities

# Create a dictionary and corpus needed for LSI
dictionary = corpora.Dictionary(tokenized_corpus)
corpus = [dictionary.doc2bow(text) for text in tokenized_corpus]

# Build the LSI model
lsi = models.LsiModel(corpus, id2word=dictionary, num_topics=300)

# Build the index
index = similarities.MatrixSimilarity(lsi[corpus])

# Query processing and retrieval using LSI
def lsi_retrieve(query, top_k=5):
    query_bow = dictionary.doc2bow(word_tokenize(query.lower()))
    query_lsi = lsi[query_bow]
    similarities = index[query_lsi]
    top_indices = np.argsort(-similarities)[:top_k]
    return top_indices, similarities[top_indices]

# Perform a query
lsi_indices, lsi_scores = lsi_retrieve(query)

# Output results
for idx, score in zip(lsi_indices, lsi_scores):
    print(f"Document index: {idx}, LSI Score: {score}\nDocument: {texts[idx][:200]}\n")

Document index: 7696, LSI Score: 0.5782680511474609
Document: From: nsmca@aurora.alaska.edu
Subject: Long Term Space Voyanges and Effect NEwsgroup?
Lines: 12
Nntp-Posting-Host: acad3.alaska.edu
Organization: University of Alaska Fairbanks

I know that alot of ho

Document index: 12480, LSI Score: 0.5225964784622192
Document: From: mfox@nyx.cs.du.edu (mark fox)
Subject: Re: Battery storage -- why not charge and store dry?
Organization: University of Denver, Dept. of Math & Comp. Sci.
Lines: 12


     Quite right, your batt

Document index: 7229, LSI Score: 0.4965341091156006
Document: From: "nigel allen" <nigel.allen@canrem.com>
Subject: HHS Secretary Shalala to Address AFT's Paraprofessional and School-Related Personnel Conference
Reply-To: "nigel allen" <nigel.allen@canrem.com>
O

Document index: 1266, LSI Score: 0.48159050941467285
Document: From: mmm@cup.portal.com (Mark Robert Thorson)
Subject: Re: centi- and milli- pedes
Organization: The Portal System (TM)
Lines: 5

I remember as

### Exercise 1

Come up with 2 different queries of your choice, each longer than 3 words, and find the best matching document for each of them using cosine similarity, BM25 and LSI.

Identify which algorithm worked best for each of your quotes.

In [19]:
queries = [
    "global economic inflation and unemployment trends",
    "space telescope images of distant galaxies"
]

def run_all_models(query):
    cos_idx, cos_scores = retrieve_documents(query, vectorizer, tfidf_matrix, top_k=1)
    bm_idx, bm_scores = bm25_retrieve(query, top_k=1)
    lsi_idx, lsi_scores = lsi_retrieve(query, top_k=1)
    return {
        "Cosine Similarity": (int(cos_idx[0]), float(cos_scores[0])),
        "BM25": (int(bm_idx[0]), float(bm_scores[0])),
        "LSI": (int(lsi_idx[0]), float(lsi_scores[0]))
    }

best_algorithm = {
    "global economic inflation and unemployment trends": "BM25",
    "space telescope images of distant galaxies": "BM25"
}

for q in queries:
    print("\n" + "=" * 90)
    print("Query:", q)
    results = run_all_models(q)
    for model_name, (doc_idx, score) in results.items():
        category = target_names[target[doc_idx]]
        preview = texts[doc_idx][:220].replace("\n", " ")
        print(f"{model_name:<18} -> Doc {doc_idx}, Score={score:.6f}, Category={category}")
        print("Preview:", preview, "...")
    print("Best overall for this query:", best_algorithm[q])

print("\nFinal answer for Exercise 1:")
print("1) Query: 'global economic inflation and unemployment trends' -> Best: BM25")
print("2) Query: 'space telescope images of distant galaxies' -> Best: BM25")


Query: global economic inflation and unemployment trends
Cosine Similarity  -> Doc 12094, Score=0.190745, Category=talk.politics.misc
Preview: From: demon@desire.wright.edu (Not a Boomer) Subject: Re: Supply Side Economic Policy Article-I.D.: desire.1993Apr6.130430.8264 Organization: ACME Products Lines: 65  In article <186042@pyramid.pyramid.com>, pcollac@pyrn ...
BM25               -> Doc 16494, Score=26.396671, Category=talk.politics.misc
Preview: From: erics@netcom.com (Eric Smith) Subject: Re: Trickle down (Was: 1937 was: Dan Quayle, genius Organization: NETCOM On-line Communication Services (408 241-9760 guest) Lines: 36  garrett@Ingres.COM  writes:  >rn11195@m ...
LSI                -> Doc 7696, Score=0.575065, Category=sci.space
Preview: From: nsmca@aurora.alaska.edu Subject: Long Term Space Voyanges and Effect NEwsgroup? Lines: 12 Nntp-Posting-Host: acad3.alaska.edu Organization: University of Alaska Fairbanks  I know that alot of how people think and a ...
Best overall for t

# Recommendation Systems

Recommendation systems are algorithms aimed at suggesting relevant items to users. These systems are fundamental in various applications, such as online shopping, streaming services, and social media platforms. They can enhance user experience and engagement by personalizing content and suggestions. Recommendation systems generally fall into two categories:

### Content-Based Filtering
This approach recommends items similar to those a user has liked in the past, based on the features of the items. For instance, in a movie recommendation system, if a user has shown a preference for action movies, the system would recommend other movies categorized as action. The similarity of items is determined through features like genre, director, description, etc.

### Collaborative Filtering
This technique makes recommendations based on the preferences of similar users. It can be divided into:

- **User-Based Collaborative Filtering**: This method finds users whose preferences are similar to the target user and recommends items they have liked. The assumption is that similar users will like similar items.

- **Item-Based Collaborative Filtering**: Rather than focusing on similarity between users, this method finds items that are similar to those the user has already liked, based on the user interaction history across the user base. (for brevity will be skipped in this laboratory)

### Key Concepts
- **User-Item Interactions**: Most systems utilize a matrix of user-item interactions, which can be explicit (e.g., ratings) or implicit (e.g., views, clicks).
- **Similarity Metrics**: Measures like cosine similarity, Pearson correlation, and Jaccard similarity are commonly used to compute the similarity between items or users.

Recommendation systems help in navigating vast information spaces by effectively filtering out less relevant content, thus improving user satisfaction and retention.

In [11]:
# @title Cosine similarity based recommendations
from sklearn.metrics.pairwise import cosine_similarity

# Assuming tfidf_matrix is the TF-IDF representation of the documents
similarity_matrix = cosine_similarity(tfidf_matrix)

print(similarity_matrix.shape)  # Should be (n_documents, n_documents)

(18846, 18846)


In [12]:
def recommend_documents(doc_index, similarity_matrix, top_k=5):
    similarity_scores = list(enumerate(similarity_matrix[doc_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    similarity_scores = similarity_scores[1:top_k+1]  # Exclude self

    print(f"Recommendations for document {doc_index}:")
    for i, score in similarity_scores:
        print(f"Document Index: {i} Score: {score}\nDocument: {texts[i][:11100]}...\n")

# Example: Recommend articles similar to document at index 100
recommend_documents(100, similarity_matrix)

Recommendations for document 100:
Document Index: 6375 Score: 0.9090628513338402
Document: From: prabhak@giga.cs.umn.edu (Satya Prabhakar)
Subject: Re: Europe vs. Muslim Bosnians
Nntp-Posting-Host: giga.cs.umn.edu
Organization: University of Minnesota, Minneapolis, CSci dept.
Lines: 20

(mohamed.s.sadek) writes:
>
>I like what Mr. Joseph Biden had to say yesterday 5/11/93 in the senate.
>
>Condemening the european lack of action and lack of support to us plans 
>and calling that "moral rape".
>
>He went on to say that the reason for that is "out right religious BIGOTRY"

Actually, this strife in Yugoslavia goes back a long way. Bosinan Muslims,
in collaboration with the Nazis, did to Serbians after the first world
war what Serbs are doing to Muslims now. This is not a fresh case of
ethnic cleansing but just another chapter in the continuing saga
of intense mutual hatred, destruction,... Not taking sides in this
perpetual war does not amount to religious bigotry. It could just
be helple

In [13]:
# @title User-based collaborative filtering
import zipfile
import pandas as pd

# Path to the zip file
zip_file_path = 'ml-100k.zip'

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall('')

# Paths to the datasets
ratings_path = 'ml-100k/u.data'
movies_path = 'ml-100k/u.item'

# Load the ratings data
ratings = pd.read_csv(ratings_path, sep='\t', header=None, names=['user_id', 'item_id', 'rating', 'timestamp'])

# Load the movie data
movies = pd.read_csv(movies_path, sep='|', header=None, names=['movie_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL', 'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'], encoding='latin-1', usecols=range(24))

# Show the first few rows of each DataFrame
display(ratings.head())
display(movies.head())

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


,movie_id,title,release_date,video_release_date,IMDb_URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [14]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Assuming ratings and movies are already loaded as described

# Create a user-item matrix
user_item_matrix = ratings.pivot_table(index='user_id', columns='item_id', values='rating')

# Replace NaN values with 0 (assuming non-rating implies a neutral sentiment)
user_item_matrix = user_item_matrix.fillna(0)

# Compute the cosine similarity between users
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

def recommend_movies(user_id, user_similarity_df, user_item_matrix, top_n=5):
    # Get similarity scores for the user in question with all other users
    sim_scores = user_similarity_df[user_id]

    # Predict scores (weighted sum of ratings)
    weighted_sum = np.dot(sim_scores, user_item_matrix)
    sim_sum = np.abs(sim_scores).sum()
    predicted_ratings = weighted_sum / sim_sum

    # Create a DataFrame of predicted ratings
    predicted_ratings = pd.Series(predicted_ratings, index=user_item_matrix.columns)

    # Filter out movies already rated by the user
    already_rated = user_item_matrix.loc[user_id]
    predicted_ratings = predicted_ratings[already_rated[already_rated == 0].index]

    # Get the top N movie IDs
    recommended_movie_ids = predicted_ratings.nlargest(top_n).index

    # Map movie IDs to titles
    recommended_movies = movies[movies['movie_id'].isin(recommended_movie_ids)]['title']
    return recommended_movies

# Example usage: Recommend movies for user with ID 72
print('Recommendations for user #72:')
recommendations = recommend_movies(72, user_similarity_df, user_item_matrix, top_n=5)
print(recommendations)

Recommendations for user #72:
21                          Braveheart (1995)
167    Monty Python and the Holy Grail (1974)
172                Princess Bride, The (1987)
182                              Alien (1979)
257                            Contact (1997)
Name: title, dtype: object


In [15]:
def counter_recommend_movies(user_id, user_similarity_df, user_item_matrix, top_n=5):
    # Get similarity scores for the user in question with all other users
    sim_scores = user_similarity_df[user_id]

    # Predict scores (weighted sum of ratings), but invert the similarity scores by subtracting from 1
    weighted_sum = np.dot(1 - sim_scores, user_item_matrix)  # Subtracting scores from 1 to invert
    sim_sum = np.abs(1 - sim_scores).sum()  # Ensure normalization by the sum of the inverted scores
    predicted_ratings = weighted_sum / sim_sum

    # Create a DataFrame of predicted ratings
    predicted_ratings = pd.Series(predicted_ratings, index=user_item_matrix.columns)

    # Filter out movies already rated by the user
    already_rated = user_item_matrix.loc[user_id]
    predicted_ratings = predicted_ratings[already_rated[already_rated == 0].index]

    # Get the top N movie IDs based on the lowest predicted ratings
    counter_recommended_movie_ids = predicted_ratings.nsmallest(top_n).index

    # Map movie IDs to titles
    counter_recommended_movies = movies[movies['movie_id'].isin(counter_recommended_movie_ids)]['title']
    return counter_recommended_movies

# Example usage: Counter-recommend movies for user with ID 196
counter_recommendations = counter_recommend_movies(72, user_similarity_df, user_item_matrix, top_n=5)
print('Counter-Recommendations for user #72:')
print(counter_recommendations)

Counter-Recommendations for user #72:
598     Police Story 4: Project S (Chao ji ji hua) (1993)
829                                       Power 98 (1995)
851                              Bloody Child, The (1996)
1620                                Butterfly Kiss (1995)
1658                      Getting Away With Murder (1996)
Name: title, dtype: object


### Exercise 2
1. Look at the movies dataframe and choose 5+ movies that you know. You can use display(movies) to interact with the dataframe.
2. Create 5+ new entries in your ratings dataframe with a new user_id (representing you) and add your ratings for the movies selected.
3. Find the movies that get recommended to you (using your newly created user id) and see if you agree with the recommendations.
4. Find the movies that are least likely to get recommended to you and see if you agree.
5. Add to your movies dataframe a column called average_rating. This column will contain the average rating given by users in the ratings dataframe to each movie. Display your movie recommendations again, this time displaying their average rating too.

Hint for 5: Calculate average rating per movie in ratings dataframe, then merge the two dataframes based on movie id. Fill NaN values with 0 if needed.

In [18]:
# 1) Choose 5+ movies and 2) add ratings with a new user_id
new_user_id = int(ratings["user_id"].max()) + 1
current_ts = int(pd.Timestamp.now().timestamp())

my_ratings = pd.DataFrame([
    {"user_id": new_user_id, "item_id": 50,  "rating": 5, "timestamp": current_ts},  # Star Wars (1977)
    {"user_id": new_user_id, "item_id": 1,   "rating": 4, "timestamp": current_ts},  # Toy Story (1995)
    {"user_id": new_user_id, "item_id": 100, "rating": 5, "timestamp": current_ts},  # Fargo (1996)
    {"user_id": new_user_id, "item_id": 96,  "rating": 4, "timestamp": current_ts},  # Terminator 2
    {"user_id": new_user_id, "item_id": 313, "rating": 3, "timestamp": current_ts},  # Titanic (1997)
    {"user_id": new_user_id, "item_id": 121, "rating": 4, "timestamp": current_ts}   # Independence Day
])

ratings_aug = pd.concat([ratings, my_ratings], ignore_index=True)

# 3) Recommend movies for new user
user_item_matrix_aug = ratings_aug.pivot_table(index="user_id", columns="item_id", values="rating").fillna(0)
user_similarity_aug = cosine_similarity(user_item_matrix_aug)
user_similarity_df_aug = pd.DataFrame(
    user_similarity_aug,
    index=user_item_matrix_aug.index,
    columns=user_item_matrix_aug.index
)

sim_scores = user_similarity_df_aug[new_user_id]
weighted_sum = np.dot(sim_scores, user_item_matrix_aug)
sim_sum = np.abs(sim_scores).sum()
predicted_ratings = pd.Series(weighted_sum / sim_sum, index=user_item_matrix_aug.columns)

already_rated = user_item_matrix_aug.loc[new_user_id]
predicted_ratings = predicted_ratings[already_rated[already_rated == 0].index]
recommended_ids = predicted_ratings.nlargest(5).index

# 4) Least likely recommendations
weighted_sum_counter = np.dot(1 - sim_scores, user_item_matrix_aug)
sim_sum_counter = np.abs(1 - sim_scores).sum()
predicted_counter = pd.Series(weighted_sum_counter / sim_sum_counter, index=user_item_matrix_aug.columns)
predicted_counter = predicted_counter[already_rated[already_rated == 0].index]
counter_ids = predicted_counter.nsmallest(5).index

# 5) Add average_rating to movies and display
avg_rating = ratings_aug.groupby("item_id")["rating"].mean().reset_index()
avg_rating.columns = ["movie_id", "average_rating"]

movies_with_avg = movies.merge(avg_rating, on="movie_id", how="left")
movies_with_avg["average_rating"] = movies_with_avg["average_rating"].fillna(0)

selected_movies = movies_with_avg[movies_with_avg["movie_id"].isin(my_ratings["item_id"])][["movie_id", "title", "average_rating"]]
recommended_movies = movies_with_avg[movies_with_avg["movie_id"].isin(recommended_ids)][["movie_id", "title", "average_rating"]]
counter_movies = movies_with_avg[movies_with_avg["movie_id"].isin(counter_ids)][["movie_id", "title", "average_rating"]]

print("My new user_id:", new_user_id)
print("\nMovies I rated:")
display(selected_movies.sort_values("movie_id"))

print("Top recommendations for me:")
display(recommended_movies.sort_values("average_rating", ascending=False))

print("Least likely recommendations for me:")
display(counter_movies.sort_values("average_rating", ascending=True))

My new user_id: 944

Movies I rated:


,movie_id,title,average_rating
0,1,Toy Story (1995),3.878587
49,50,Star Wars (1977),4.359589
95,96,Terminator 2: Judgment Day (1991),4.006757
99,100,Fargo (1996),4.157171
120,121,Independence Day (ID4) (1996),3.439535
312,313,Titanic (1997),4.242165


Top recommendations for me:


,movie_id,title,average_rating
126,127,"Godfather, The (1972)",4.283293
173,174,Raiders of the Lost Ark (1981),4.252381
180,181,Return of the Jedi (1983),4.007890
257,258,Contact (1997),3.803536
6,7,Twelve Monkeys (1995),3.798469


Least likely recommendations for me:


,movie_id,title,average_rating
1600,1601,Office Killer (1997),1.0
1625,1626,Nobody Loves Me (Keiner liebt mich) (1994),1.0
1653,1654,Chairman of the Board (1998),1.0
1658,1659,Getting Away With Murder (1996),1.0
1660,1661,"New Age, The (1994)",1.0
